# Experiment 0B: SHD — Input-Layer Perturbation (Original Beyond Rate Reproduction)

## Overview

This notebook reproduces the **original Beyond Rate** SHD experiment, in which
perturbation is applied to the **input spike trains** (not the hidden layer).
Its purpose is to validate that our pipeline (training, perturbation,
evaluation) matches the published results before drawing conclusions from
the hidden-layer experiments in Phases 1–4.

**Key idea (original Beyond Rate):** For each value of $f \in [0, 1]$, we
*regenerate* the train/val/test data by relocating a fraction $f$ of each
input neuron's spikes to random empty time bins (preserving per-neuron spike
counts). A fresh model is trained on each perturbed dataset and evaluated on
the corresponding perturbed test set.

**Architecture:** Input → 128 hidden → 128 hidden → 20 output (SRMALPHA)

**Dataset variants:** whole (700 input neurons), part (224), norm (224)

A global `USE_DELAY` flag selects whether to train with learnable delays
(SGD-delay) or without (SGD).

## 1. Imports and Setup

In [ ]:
import os
import json
import random
from typing import Optional

import numpy as np
from scipy.io import loadmat
import matplotlib.pyplot as plt
import torch
from torch import nn
from tqdm import tqdm

import slayerSNN as snn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Global Configuration

All key hyper-parameters and switches live here. Switching between dataset
variants or delay/no-delay mode requires editing only this cell.

In [ ]:
# =====================================================================
# Network variant: True for SGD-delay, False for SGD (no delay)
# =====================================================================
USE_DELAY: bool = True

# =====================================================================
# Dataset variant: "whole", "part", or "norm"
# =====================================================================
DATASET_KEY: str = "whole"

# --- Dataset configurations ---
DATASET_CONFIGS = {
    "whole": {"mat_file": "shd_data/shd_whole.mat", "input_dim": 700},
    "part":  {"mat_file": "shd_data/shd_part_new.mat", "input_dim": 224},
    "norm":  {"mat_file": "shd_data/shd_norm_new.mat", "input_dim": 224},
}

# --- SLAYER neuron and simulation descriptors (match original shd_train.py) ---
SIM_PARAMS = {"Ts": 1, "tSample": 200}
LIF_PARAMS = {
    "type": "SRMALPHA",
    "theta": 10,
    "tauSr": 1,
    "tauRho": 0.1,
    "tauRef": 2,
    "scaleRef": 2,
    "scaleRho": 0.1,
}

# --- Data split ratios (match original) ---
TRAIN_RANGE = (0.0, 0.6)
VAL_RANGE = (0.6, 0.75)
TEST_RANGE = (0.75, 0.9)

# --- Training hyper-parameters ---
HIDDEN_UNITS: int = 128
NUM_CLASSES: int = 20
EPOCHS: int = 800
BATCH_SIZE: int = 128
LEARNING_RATE: float = 0.1
SEED: int = 42
MAX_DELAY: int = 64
EARLY_STOP_PATIENCE: int = 300

# --- Input-perturbation sweep ---
F_VALUES: list[float] = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

# --- Derived names ---
INPUT_DIM: int = DATASET_CONFIGS[DATASET_KEY]["input_dim"]
MAT_FILE: str = DATASET_CONFIGS[DATASET_KEY]["mat_file"]
DELAY_TAG: str = "delay" if USE_DELAY else "nodelay"
MODEL_PREFIX: str = f"shd_{DATASET_KEY}_{DELAY_TAG}_inputpert"

print(f"Dataset: {DATASET_KEY} | Input dim: {INPUT_DIM}")
print(f"Network mode: {'SGD-delay' if USE_DELAY else 'SGD (no delay)'}")
print(f"Model prefix: {MODEL_PREFIX}")
print(f"f sweep: {F_VALUES}")

## 3. Load SHD Dataset

Load the dense spike-train dataset from the local `.mat` file. Each sample
has shape `(num_neurons, T)` with binary spike values. Time dimension is
padded to 200 time steps to match `tSample`.

In [ ]:
def load_shd_data(mat_path: str, target_T: int = 200) -> tuple[np.ndarray, np.ndarray]:
    """Load SHD dataset from a .mat file and pad time dimension.

    Args:
        mat_path: Path to the .mat file containing 'X' and 'Y'.
        target_T: Target time dimension (zero-pad if shorter).

    Returns:
        Tuple of (X, Y) where X has shape (N, neurons, target_T).
    """
    data = loadmat(mat_path)
    X = data["X"]
    Y = data["Y"].ravel()

    n_samples, n_neurons, T = X.shape
    if T < target_T:
        padded = np.zeros((n_samples, n_neurons, target_T), dtype=X.dtype)
        padded[:, :, :T] = X
        X = padded
        print(f"Padded time dimension from {T} to {target_T}")

    print(f"Loaded {mat_path}: X={X.shape}, Y={Y.shape}, classes={len(np.unique(Y))}")
    return X, Y


X_all, Y_all = load_shd_data(MAT_FILE, target_T=SIM_PARAMS["tSample"])

## 4. Input-Layer Spike Perturbation

The original Beyond Rate perturbation. For each input neuron, a fraction
$f$ of spikes is removed and re-placed at uniformly random empty time bins.
Per-neuron spike counts are preserved (the rate code is preserved); only
spike timing is destroyed.

In [ ]:
def partial_randomize_spike_train(
    spike_train: np.ndarray,
    f: float = 0.0,
    max_attempts: int = 50,
) -> np.ndarray:
    """Randomly relocate a fraction *f* of each neuron's spikes.

    Args:
        spike_train: Binary array of shape (num_neurons, T).
        f: Fraction of spikes to relocate (0 = no change, 1 = full shuffle).
        max_attempts: Max tries to find an empty time bin per spike.

    Returns:
        Perturbed spike train with same shape and per-neuron spike counts.
    """
    if f <= 0:
        return spike_train

    num_neurons, T = spike_train.shape
    new_train = np.copy(spike_train)

    for neuron_idx in range(num_neurons):
        spike_times = np.where(spike_train[neuron_idx] == 1)[0]
        for old_time in spike_times:
            if np.random.rand() < f:
                new_train[neuron_idx, old_time] = 0
                inserted = False
                attempts = 0
                while not inserted and attempts < max_attempts:
                    attempts += 1
                    new_t = np.random.randint(0, T)
                    if new_train[neuron_idx, new_t] == 0:
                        new_train[neuron_idx, new_t] = 1
                        inserted = True
    return new_train


def preprocess_full_dataset(
    X: np.ndarray,
    Y: np.ndarray,
    f: float,
    training_range: tuple[float, float] = TRAIN_RANGE,
    validation_range: tuple[float, float] = VAL_RANGE,
    testing_range: tuple[float, float] = TEST_RANGE,
    seed: int = 42,
) -> tuple:
    """Split dataset and apply input-spike perturbation at level *f* to all splits.

    Mirrors the preprocessing in the original `shd_train.py`.
    """
    np.random.seed(seed)

    N = len(Y)
    train_idx = np.arange(int(N * training_range[0]), int(N * training_range[1]))
    val_idx = np.arange(int(N * validation_range[0]), int(N * validation_range[1]))
    test_idx = np.arange(int(N * testing_range[0]), int(N * testing_range[1]))

    np.random.shuffle(train_idx)

    def _process(indices: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        X_out = np.array([partial_randomize_spike_train(X[i], f) for i in indices])
        return X_out, Y[indices]

    return _process(train_idx), _process(val_idx), _process(test_idx)

## 5. Network Architecture

A 2-hidden-layer SLAYER SNN matching the original Beyond Rate SHD code:

```
Input → [PSP + fc1 + spike] → (delay1) → hidden1 spikes
      → [PSP + fc2 + spike] → (delay2) → hidden2 spikes
      → [PSP + fc3 + spike] → output
```

In [ ]:
class SHDNetwork(nn.Module):
    """2-hidden-layer SLAYER SNN for SHD classification."""

    def __init__(
        self,
        input_dim: int,
        hidden_units: int = 128,
        num_classes: int = 20,
        use_delay: bool = True,
        max_delay: int = 64,
    ):
        super().__init__()
        slayer = snn.layer(LIF_PARAMS, SIM_PARAMS)
        self.slayer = slayer
        self.use_delay = use_delay
        self.max_delay = max_delay

        self.fc1 = nn.utils.weight_norm(
            slayer.dense(input_dim, hidden_units), name="weight"
        )
        self.fc2 = nn.utils.weight_norm(
            slayer.dense(hidden_units, hidden_units), name="weight"
        )
        self.fc3 = nn.utils.weight_norm(
            slayer.dense(hidden_units, num_classes), name="weight"
        )

        if use_delay:
            self.delay1 = slayer.delay(hidden_units)
            self.delay2 = slayer.delay(hidden_units)

    def _prepare_input(self, x: torch.Tensor) -> torch.Tensor:
        """Ensure input is 5-D NCHWT on the correct device."""
        if isinstance(x, np.ndarray):
            x = torch.from_numpy(x)
        if x.dim() == 3:
            x = x.unsqueeze(2).unsqueeze(3)
        return x.float().to(device)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Standard forward pass (perturbation is applied to the input upstream)."""
        x = self._prepare_input(x)
        x = self.slayer.spike(self.fc1(self.slayer.psp(x)))
        if self.use_delay:
            x = self.delay1(x)
        x = self.slayer.spike(self.fc2(self.slayer.psp(x)))
        if self.use_delay:
            x = self.delay2(x)
        x = self.slayer.spike(self.fc3(self.slayer.psp(x)))
        return x

    def clamp_delays(self, max1: int = 64, max2: int = 64) -> None:
        if not self.use_delay:
            return
        self.delay1.delay.data.clamp_(0, max1)
        self.delay2.delay.data.clamp_(0, max2)

    def get_delays(self) -> dict[str, np.ndarray]:
        delays: dict[str, np.ndarray] = {}
        if self.use_delay:
            delays["delay1"] = self.delay1.delay.data.cpu().numpy()
            delays["delay2"] = self.delay2.delay.data.cpu().numpy()
        return delays

## 6. Training Loop

Train one model per *f* on the **input-perturbed** training set. Uses
SLAYER's `SpikeRate` loss (matching the original `shd_train.py`),
Nadam optimiser, multi-step LR schedule, and the original adaptive
delay-clamping logic.

In [ ]:
def set_seed(seed: int) -> None:
    """Set random seeds for reproducibility."""
    import torch.backends.cudnn as cudnn
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        cudnn.benchmark = False
        cudnn.deterministic = True
        cudnn.enabled = False


def build_loss_and_optimizer(net: SHDNetwork, lr: float = 0.1) -> tuple:
    """Build SpikeRate loss, Nadam optimizer, and LR scheduler.

    Matches the original `shd_train.py` (SpikeRate, not NumSpikes).
    """
    error_cfg = {
        "neuron": LIF_PARAMS,
        "simulation": SIM_PARAMS,
        "training": {
            "error": {
                "type": "SpikeRate",
                "tgtSpikeRegion": {"start": 0, "stop": 200},
                "tgtSpikeRate": {True: 0.2, False: 0.02},
            }
        },
    }
    loss_fn = snn.spikeLoss.spikeLoss(error_cfg)
    optimizer = snn.utils.optim.Nadam(net.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer, milestones=[300], gamma=0.1
    )
    return loss_fn, optimizer, scheduler


def train_model(
    X_train: np.ndarray,
    Y_train: np.ndarray,
    X_val: np.ndarray,
    Y_val: np.ndarray,
    input_dim: int,
    hidden_units: int = 128,
    num_classes: int = 20,
    use_delay: bool = True,
    max_delay: int = 64,
    epochs: int = 1000,
    batch_size: int = 128,
    lr: float = 0.1,
    seed: int = 42,
    patience: int = 300,
) -> tuple[SHDNetwork, dict]:
    """Train an SHDNetwork on the (already-perturbed) training data."""
    set_seed(seed)

    net = SHDNetwork(
        input_dim, hidden_units, num_classes, use_delay, max_delay
    ).to(device)
    loss_fn, optimizer, scheduler = build_loss_and_optimizer(net, lr=lr)
    loss_fn = loss_fn.to(device)

    best_val_loss = float("inf")
    best_model_state: Optional[dict] = None
    early_stop_counter = 0

    update1 = 0
    update2 = 0
    thea1 = max_delay
    thea2 = max_delay

    log = {
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
        "val_acc": [],
        "delay_mean": [],
    }

    n_train = len(Y_train)
    total_steps = epochs * max(1, n_train // batch_size)
    with tqdm(total=total_steps, desc="Training") as pbar:
        for epoch in range(epochs):
            net.train()
            indices = np.arange(n_train)
            np.random.shuffle(indices)
            batch_losses = []

            for b in range(0, n_train, batch_size):
                batch_idx = indices[b:b + batch_size]
                xb = X_train[batch_idx]
                yb = Y_train[batch_idx]

                x = torch.tensor(xb).unsqueeze(2).unsqueeze(3).float().to(device)
                y = torch.tensor(yb).long().to(device)
                target = torch.zeros(
                    (len(y), num_classes, 1, 1, 1), device=device
                )
                target.scatter_(1, y[:, None, None, None, None], 1.0)

                outputs = net(x)
                loss = loss_fn.spikeRate(outputs, target)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                batch_losses.append(loss.item())
                pbar.update(1)

            # Adaptive delay clamping (matches original shd_train.py)
            if use_delay:
                if epoch <= 250:
                    net.clamp_delays(max_delay, max_delay)
                else:
                    update1 += 1
                    update2 += 1
                    for name, param in net.named_parameters():
                        if "delay1.delay" in name and update1 > 150:
                            sorted_ = torch.sort(
                                torch.floor(param.detach().flatten())
                            )[0]
                            thea1_val = torch.max(sorted_)
                            if sorted_[108] > (thea1_val - 5):
                                thea1 = int(thea1_val.item()) + 1
                                update1 = 0
                        elif "delay2.delay" in name and update2 > 150:
                            sorted_ = torch.sort(
                                torch.floor(param.detach().flatten())
                            )[0]
                            thea2_val = torch.max(sorted_)
                            if sorted_[108] > (thea2_val - 5):
                                thea2 = int(thea2_val.item()) + 1
                                update2 = 0
                    net.clamp_delays(thea1, thea2)

            # Validation
            net.eval()
            val_loss = 0.0
            correct = 0
            total = 0
            n_val_batches = 0
            with torch.no_grad():
                for b in range(0, len(Y_val), batch_size):
                    xb = X_val[b:b + batch_size]
                    yb = Y_val[b:b + batch_size]
                    x = torch.tensor(xb).unsqueeze(2).unsqueeze(3).float().to(device)
                    y = torch.tensor(yb).long().to(device)
                    target = torch.zeros(
                        (len(y), num_classes, 1, 1, 1), device=device
                    )
                    target.scatter_(1, y[:, None, None, None, None], 1.0)
                    outputs = net(x)
                    val_loss += loss_fn.spikeRate(outputs, target).item()
                    pred = snn.predict.getClass(outputs)
                    correct += (pred.cpu() == y.cpu()).sum().item()
                    total += len(y)
                    n_val_batches += 1
            val_loss /= max(1, n_val_batches)
            val_acc = correct / max(1, total)
            train_loss = float(np.mean(batch_losses))

            delays = net.get_delays()
            avg_delay = (
                float(np.mean([np.mean(d) for d in delays.values() if len(d) > 0]))
                if delays else 0.0
            )

            log["epoch"].append(epoch)
            log["train_loss"].append(train_loss)
            log["val_loss"].append(float(val_loss))
            log["val_acc"].append(float(val_acc))
            log["delay_mean"].append(avg_delay)

            pbar.set_postfix(
                epoch=epoch + 1,
                train=f"{train_loss:.3f}",
                val=f"{val_loss:.3f}",
                acc=f"{val_acc:.2%}",
                delay=f"{avg_delay:.1f}",
            )
            scheduler.step()

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_model_state = {k: v.clone() for k, v in net.state_dict().items()}
                early_stop_counter = 0
            else:
                early_stop_counter += 1
                if early_stop_counter >= patience:
                    print(f"\nEarly stopping at epoch {epoch + 1}")
                    break

    if best_model_state is not None:
        net.load_state_dict(best_model_state)

    return net, log

## 7. Test-Set Evaluation

Evaluate the trained model on the (input-perturbed) test set. No further
perturbation is applied inside the network — the perturbation already lives
in the data.

In [ ]:
def test_accuracy(
    net: SHDNetwork,
    X_test: np.ndarray,
    Y_test: np.ndarray,
    batch_size: int = 64,
) -> float:
    """Compute classification accuracy on a (pre-perturbed) test set."""
    net.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for b in range(0, len(Y_test), batch_size):
            xb = X_test[b:b + batch_size]
            yb = Y_test[b:b + batch_size]
            x = torch.tensor(xb).unsqueeze(2).unsqueeze(3).float().to(device)
            pred = snn.predict.getClass(net(x))
            correct += int((pred.cpu().numpy() == yb).sum())
            total += len(yb)
    return correct / max(1, total)

## 8. Run: Sweep Over Input-Perturbation Levels

For each $f$, regenerate the (perturbed) train/val/test splits, train a
fresh model, and record the test accuracy. This is the main loop of
the original Beyond Rate SHD experiment.

In [ ]:
os.makedirs("data", exist_ok=True)
os.makedirs("log", exist_ok=True)

sweep_results: dict[float, dict] = {}
training_logs: dict[str, dict] = {}

for f in F_VALUES:
    f = float(f)
    print(f"\n===== Input perturbation f = {f} =====")

    print("Preprocessing dataset (perturbing input spike trains)...")
    (X_train, Y_train), (X_val, Y_val), (X_test, Y_test) = preprocess_full_dataset(
        X_all, Y_all, f,
        training_range=TRAIN_RANGE,
        validation_range=VAL_RANGE,
        testing_range=TEST_RANGE,
        seed=SEED,
    )
    print(f"Train: {len(Y_train)} | Val: {len(Y_val)} | Test: {len(Y_test)}")

    net, log = train_model(
        X_train, Y_train, X_val, Y_val,
        input_dim=INPUT_DIM,
        hidden_units=HIDDEN_UNITS,
        num_classes=NUM_CLASSES,
        use_delay=USE_DELAY,
        max_delay=MAX_DELAY,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LEARNING_RATE,
        seed=SEED,
        patience=EARLY_STOP_PATIENCE,
    )

    acc = test_accuracy(net, X_test, Y_test, batch_size=64)
    print(f"Test accuracy at f={f}: {acc:.4f}")

    sweep_results[f] = {"accuracy": float(acc)}
    training_logs[f"f={f}"] = log

    model_path = f"data/{MODEL_PREFIX}_f{int(f * 10):02d}.pt"
    torch.save(net.state_dict(), model_path)
    print(f"Saved model to {model_path}")

## 9. Plot: Accuracy vs Input-Perturbation Level

In [ ]:
def plot_input_perturbation_curve(results: dict[float, dict]) -> None:
    """Plot test accuracy vs input perturbation level f."""
    f_vals = sorted(results.keys())
    accs = [results[f]["accuracy"] for f in f_vals]

    color = "tab:orange" if USE_DELAY else "tab:blue"
    label = (
        f"SGD-delay ({DATASET_KEY})"
        if USE_DELAY
        else f"SGD ({DATASET_KEY})"
    )

    plt.figure(figsize=(8, 5))
    plt.plot(f_vals, accs, "o-", color=color, label=label)
    plt.xlabel("Input Perturbation Level (f)")
    plt.ylabel("Test Accuracy")
    plt.title(
        f"Exp 0B: SHD {DATASET_KEY} "
        f"\u2014 Accuracy vs Input-Layer Perturbation ({DELAY_TAG})"
    )
    plt.ylim(0, 1.05)
    plt.grid(True, alpha=0.3)
    plt.legend()

    for f_val, acc in zip(f_vals, accs):
        plt.annotate(
            f"{acc:.3f}",
            (f_val, acc),
            textcoords="offset points",
            xytext=(0, 12),
            ha="center",
            fontsize=9,
        )

    plt.tight_layout()
    fig_path = (
        f"log/shd_{DATASET_KEY}_{DELAY_TAG}_input_perturbation.png"
    )
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Figure saved to {fig_path}")


plot_input_perturbation_curve(sweep_results)

## 10. Save Sweep Results and Training Logs

In [ ]:
results_serialisable = {
    str(f_val): {"accuracy": float(d["accuracy"])}
    for f_val, d in sweep_results.items()
}

results_path = (
    f"log/shd_{DATASET_KEY}_{DELAY_TAG}_input_perturbation_results.json"
)
with open(results_path, "w") as fp:
    json.dump(results_serialisable, fp, indent=2)
print(f"Results saved to {results_path}")

logs_serialisable = {
    f_key: {
        k: [float(v) for v in vals] if isinstance(vals, list) else vals
        for k, vals in log.items()
    }
    for f_key, log in training_logs.items()
}
logs_path = f"log/shd_{DATASET_KEY}_{DELAY_TAG}_input_perturbation_training_logs.json"
with open(logs_path, "w") as fp:
    json.dump(logs_serialisable, fp, indent=2)
print(f"Training logs saved to {logs_path}")

## 11. Comparison Against the Paper

**Reference targets (Beyond Rate, SHD):**
- Test accuracy at $f = 0$ around the published SHD numbers for SGD vs SGD-delay.
- Monotonic accuracy decay with $f$.
- SGD-delay degrades faster than SGD if it relies more on input timing.

Run this notebook for both `USE_DELAY = True` and `USE_DELAY = False`, then
overlay the resulting curves to compare against the paper figures and
decide whether the pipeline is trusted (Phase 0 success criteria).